# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by the Croissant metadata.

In [ ]:
from pprint import pprint

# List record sets and each field (with their @id) for context
record_sets = getattr(metadata, 'record_set', [])
# If there's only one record set or single dict, wrap in list
if isinstance(record_sets, dict):
    record_sets = [record_sets]

record_set_ids = []

if not record_sets:
    # Try to auto-infer record_sets from the datasets
    print("No explicit 'record_set' property found in the schema.\nAttempting to access record set(s) via distribution field...")
    record_sets = getattr(metadata, 'distribution', [])

for rec in record_sets:
    rs_id = rec.get('@id', None) if isinstance(rec, dict) else rec
    if rs_id is not None:
        record_set_ids.append(rs_id)
        print(f"Record set @id: {rs_id}")
        # Try print columns/fields for this record set
        # For Croissant v1.0, distribution is DataDownload, not actual RecordSet (but may point to file)

# Display all available fields for each record set, if possible:
def try_fields(record_set_id):
    try:
        # Print the first 1-2 records and list keys
        for i, rec in enumerate(dataset.records(record_set=record_set_id)):
            print(f"Sample record from {record_set_id}, record {i+1}:")
            pprint(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not extract records for {record_set_id}: {e}")

for rsid in record_set_ids:
    try_fields(rsid)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# We'll use the first record set found (main data table)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

dataframes = {}
for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    if len(records) > 0:
        dataframes[record_set] = pd.DataFrame(records)
    else:
        print(f"No records found for {record_set}.")

if main_record_set_id in dataframes:
    print(f"Fields/columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"Data unavailable for main record set {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> **Note:** All field names and columns are referenced by their `@id` (column name exactly as in the DataFrame above, which should correspond to the field or column `@id`).


In [ ]:
import numpy as np

# Inspect the columns to determine numeric fields for EDA
df = dataframes.get(main_record_set_id)
if df is not None:
    # Try to auto-select a numeric field: look for typical mass spec/quant fields
    candidate_numeric_ids = [c for c in df.columns if df[c].dtype in [np.float64, np.float32, np.int32, np.int64]]
    if not candidate_numeric_ids:
        # Try to coerce columns containing numbers
        candidate_numeric_ids = []
        for c in df.columns:
            try:
                converted = pd.to_numeric(df[c].dropna(), errors='coerce')
                if converted.notnull().mean() > 0.6:
                    candidate_numeric_ids.append(c)
            except Exception:
                pass
    print(f"Numeric columns inferred: {candidate_numeric_ids}")
    if candidate_numeric_ids:
        # We'll use the first numeric field for demonstration
        numeric_field_id = candidate_numeric_ids[0]
        print(f"Using numeric field with @id: {numeric_field_id}")
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)  # 75th percentile as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:0.3f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a likely categorical field
        possible_group_fields = [c for c in df.columns if 'group' in c.lower() or 'type' in c.lower() or 'label' in c.lower() or 'sample' in c.lower()]
        group_field_id = possible_group_fields[0] if possible_group_fields else None
        if group_field_id and group_field_id in df.columns:
            print(f"Grouping by {group_field_id} (@id)")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No appropriate group field found for grouping.")
    else:
        print("No valid numeric fields available in the record set.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and candidate_numeric_ids:
    field = numeric_field_id
    plt.figure(figsize=(8,4))
    sns.histplot(df[field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {field} (@id)")
    plt.xlabel(field)
    plt.show()

    # If possible, boxplot by group_field_id
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=field)
        plt.title(f"{field} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable dataframe or numeric fields for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using the `mlcroissant` library. We loaded metadata, examined available record sets and fields by `@id`, extracted data for analysis, filtered and normalized numeric data fields, and performed basic grouping and visualization. For advanced analytics, refer to the dataset record and column/field `@id` definitions as documented in the Croissant schema.
